In [3]:
import pandas as pd
base_grande=pd.read_excel("basezona.xlsx")
base_grande

,Commodity,Country,Attribute,2000/2001,2001/2002,2002/2003,2003/2004,2004/2005,2005/2006,2006/2007,...,2017/2018,2018/2019,2019/2020,2020/2021,2021/2022,2022/2023,2023/2024,2024/2025,2025/2026,Unit Description
0,Orange Juice,Brazil,Production,978,"1,354,000","1,151,000","1,482,000","1,285,000","1,440,000","1,480,000",...,"1,004,000","1,324,000",938,944,"1,135,000","1,080,337",828.755,"1,012,840","1,032,076",(MT)
1,Orange Juice,Brazil,Imports,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,(MT)
2,Orange Juice,Brazil,Exports,"1,075,000","1,250,000","1,317,000","1,417,000","1,345,000","1,415,000","1,298,000",...,989,"1,120,000","1,036,000","1,010,000","1,068,000","1,006,167",777.025,953.84,973.276,(MT)
3,Orange Juice,Brazil,Domestic Consumption,15,15,18,20,23,28,31,...,40,52,63,70,73,75,55,58,58.7,(MT)
4,Orange Juice,Brazil,Ending Stocks,151,240,56,101,18,15,166,...,160,312,151,15,9,8.17,4.9,5.9,6,(MT)
5,Orange Juice,World,Production,"2,194,893","2,607,009","2,220,823","2,728,810","2,220,741","2,394,805","2,428,216",...,"1,578,760","2,089,376","1,469,539","1,559,158","1,688,377","1,443,469","1,239,220","1,340,308","1,351,218",(MT)
6,Orange Juice,World,Imports,519.704,"1,310,549","1,545,176","1,506,226","1,592,933","1,544,084","1,519,255",...,"1,628,498","1,506,613","1,379,265","1,356,380","1,317,015","1,346,981","1,279,618","1,044,588","1,089,000",(MT)
7,Orange Juice,World,Exports,"1,374,044","1,505,545","1,484,219","1,590,759","1,563,257","1,640,893","1,536,836",...,"1,489,867","1,604,844","1,425,538","1,464,352","1,502,111","1,365,328","1,128,012","1,238,012","1,272,546",(MT)
8,Orange Juice,World,Domestic Consumption,"1,393,523","2,332,787","2,468,936","2,518,892","2,475,337","2,396,403","2,351,464",...,"1,747,985","1,691,967","1,683,944","1,648,533","1,598,143","1,458,416","1,353,114","1,123,942","1,168,392",(MT)
9,Orange Juice,World,Ending Stocks,732.485,811.711,624.555,749.94,525.02,426.613,485.784,...,476.584,775.762,515.084,317.737,222.875,189.581,227.293,250.235,249.515,(MT)


In [5]:
import pandas as pd
import numpy as np

# 1. Carregar a base (substitua pelo caminho do seu arquivo)
df = pd.read_excel('basezona.xlsx')

# Simulando o carregamento para o script
data = {
    'Commodity': ['Orange Juice'] * 6,
    'Country': ['Brazil', 'Brazil', 'Brazil', 'Brazil', 'World', 'World'], # Reduzido para focar na lógica
    'Attribute': ['Production', 'Imports', 'Exports', 'Domestic Consumption', 'Production', 'Exports'],
    '2000/2001': ['978', '0', '1,075,000', '15', '2,194,893', '1,500,000'],
    '2001/2002': ['1,354,000', '0', '1,250,000', '15', '2,607,009', '1,600,000'],
    '2019/2020': ['938', '0', '1,036,000', '63', '1,469,539', '1,100,000'],
    '2024/2025': ['1,012,840', '0', '953.84', '58', '1,340,308', '900.5']
}
df = pd.DataFrame(data)

# 2. Transformar de Wide para Long (Unpivot)
# Mantemos as colunas identificadoras fixas e "derretemos" os anos
id_vars = ['Commodity', 'Country', 'Attribute']
anos_cols = [col for col in df.columns if '/' in col] # Pega as colunas de ano
df_long = pd.melt(df, id_vars=id_vars, value_vars=anos_cols, 
                  var_name='Safra', value_name='Valor')

# 3. Limpeza das strings e conversão para número
# Remove vírgulas de separação de milhar e converte para float
df_long['Valor'] = df_long['Valor'].astype(str).str.replace(',', '')
df_long['Valor'] = pd.to_numeric(df_long['Valor'], errors='coerce')

# 4. Normalização da Escala (Heurística de Correção)
# Se o valor for menor que 10.000 (e não for zero), significa que ele está em "Milhares de Toneladas"
# Precisamos multiplicar por 1000 para uniformizar tudo em Toneladas Métricas (MT) absolutas
mascara_escala = (df_long['Valor'] > 0) & (df_long['Valor'] < 10000)
df_long.loc[mascara_escala, 'Valor'] = df_long.loc[mascara_escala, 'Valor'] * 1000

# 5. Tratamento de Data para o Modelo (Converter "2000/2001" para Data)
# Vamos extrair o primeiro ano da safra para servir de índice temporal (ex: "2000")
df_long['Ano'] = df_long['Safra'].str.split('/').str[0].astype(int)

# 6. Pivotar para formato analítico de Machine Learning / Econometria
# Agora cada linha é um Ano/País, e os atributos viram colunas (Features)
df_final = df_long.pivot_table(index=['Ano', 'Country'], 
                               columns='Attribute', 
                               values='Valor', 
                               aggfunc='sum').reset_index()

# Preencher possíveis valores nulos com 0 (ex: Imports)
df_final = df_final.fillna(0)

df_final

Attribute,Ano,Country,Domestic Consumption,Exports,Imports,Production
0,2000,Brazil,15000.0,1075000.0,0.0,978000.0
1,2000,World,0.0,1500000.0,0.0,2194893.0
2,2001,Brazil,15000.0,1250000.0,0.0,1354000.0
3,2001,World,0.0,1600000.0,0.0,2607009.0
4,2019,Brazil,63000.0,1036000.0,0.0,938000.0
5,2019,World,0.0,1100000.0,0.0,1469539.0
6,2024,Brazil,58000.0,953840.0,0.0,1012840.0
7,2024,World,0.0,900500.0,0.0,1340308.0


In [1]:
"""
=============================================================================
DESAFIO ESTRATÉGICO L.E.K. CONSULTING 2026 - FASE 1
Monte Carlo: Previsão de Demanda e Preço das 5 Culturas Agrícolas
=============================================================================
Culturas analisadas:
  - Milho          → Corn         (USDA PSD)
  - Cana-de-açúcar → Sugar, Centrifugal
  - Café (Arábica) → Coffee, Green
  - Citros (OJ)    → Orange Juice
  - Trigo          → Wheat

Preços de referência (externos – ver nota abaixo):
  Milho  : CBOT  (US$/bu)
  Café   : ICE   (US$/lb)
  Açúcar : ICE   (US$/lb) – raw sugar #11
  Trigo  : CBOT  (US$/bu)
  Citros : ICE OJ (US$/lb)

NOTA SOBRE BASES DE PREÇO:
  A basezona.xlsx fornece dados de volumes (USDA PSD).
  Preços históricos precisam ser baixados de fontes externas
  (CBOT, ICE, Bloomberg). O script usa preços de referência
  histórica embutidos (2005-2024) para rodar sem acesso a API.
  Para produção, substitua `PRICES_HIST` com dados reais.
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from pathlib import Path
import json

# ──────────────────────────────────────────────────────────────────────────────
# 0. CONFIGURAÇÕES GLOBAIS
# ──────────────────────────────────────────────────────────────────────────────
np.random.seed(42)
N_SIMUL   = 10_000   # simulações Monte Carlo
HORIZON   = 5        # anos de projeção (2025-2029)
PROJ_YEAR = 2024     # último ano histórico observado

INPUT_FILE  = "basezona.xlsx"
OUTPUT_DIR  = Path("/mnt/user-data/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Mapeamento nome interno → rótulo do desafio
CROP_MAP = {
    "Corn"              : "Milho",
    "Coffee, Green"     : "Café (Arábica)",
    "Sugar, Centrifugal": "Cana-de-açúcar",
    "Orange Juice"      : "Citros (OJ)",
    "Wheat"             : "Trigo",
}

# Unidades de demanda e preço por cultura
UNITS = {
    "Milho"            : {"demand": "1.000 MT",  "price": "US$/bu",  "price_factor": 1.0},
    "Café (Arábica)"   : {"demand": "M sacos 60kg","price": "US$/lb","price_factor": 1.0},
    "Cana-de-açúcar"   : {"demand": "1.000 MT",  "price": "US$/lb",  "price_factor": 1.0},
    "Citros (OJ)"      : {"demand": "1.000 MT",  "price": "US$/lb",  "price_factor": 1.0},
    "Trigo"            : {"demand": "1.000 MT",  "price": "US$/bu",  "price_factor": 1.0},
}

# ──────────────────────────────────────────────────────────────────────────────
# 1. PREÇOS HISTÓRICOS DE REFERÊNCIA (annual avg, US$)
#    Fonte: USDA ERS / ICE / CBOT (compilação pública 2005-2024)
#    Para uso real: substitua por dados reais da API
# ──────────────────────────────────────────────────────────────────────────────
PRICES_HIST = {
    # Ano : (Milho CBOT, Café ICE, Açúcar ICE, Citros ICE-OJ, Trigo CBOT)
    #        US$/bu      US$/lb     US$/lb       US$/lb        US$/bu
    2005: (2.00, 1.13, 0.098, 0.74, 3.42),
    2006: (3.04, 1.06, 0.122, 0.85, 3.56),
    2007: (3.40, 1.10, 0.098, 0.80, 4.26),
    2008: (5.74, 1.22, 0.120, 0.80, 7.57),
    2009: (3.55, 1.13, 0.189, 0.96, 5.00),
    2010: (3.69, 1.48, 0.228, 1.01, 5.70),
    2011: (6.01, 2.52, 0.264, 1.35, 7.77),
    2012: (6.98, 1.82, 0.196, 1.23, 7.77),
    2013: (4.62, 1.30, 0.170, 1.28, 6.87),
    2014: (3.61, 1.94, 0.155, 1.60, 5.99),
    2015: (3.61, 1.23, 0.120, 1.05, 4.89),
    2016: (3.36, 1.44, 0.188, 1.20, 4.64),
    2017: (3.36, 1.32, 0.148, 1.44, 4.72),
    2018: (3.61, 1.09, 0.125, 1.50, 5.16),
    2019: (3.56, 1.00, 0.126, 1.02, 4.65),
    2020: (3.56, 1.23, 0.128, 1.06, 5.05),
    2021: (5.63, 1.93, 0.175, 1.26, 7.77),
    2022: (6.70, 2.26, 0.195, 1.57, 8.86),
    2023: (4.82, 1.80, 0.232, 1.44, 6.08),
    2024: (4.35, 3.20, 0.218, 1.50, 5.75),
}

PRICE_COLS = ["Milho", "Café (Arábica)", "Cana-de-açúcar", "Citros (OJ)", "Trigo"]

# ──────────────────────────────────────────────────────────────────────────────
# 2. TRATAMENTO DA BASE (USDA PSD)
# ──────────────────────────────────────────────────────────────────────────────

def load_and_clean_base(path: str) -> dict:
    """
    Carrega basezona.xlsx, converte colunas de anos para numérico e
    retorna dict {commodity: {country: DataFrame(wide → long)}}.
    """
    df_raw = pd.read_excel(path, sheet_name="psd (3)")

    # Colunas de anos (formato 'YYYY/YYYY')
    year_cols = [c for c in df_raw.columns if "/" in str(c)]

    # Normaliza valores: remove vírgulas, converte para float
    for col in year_cols:
        df_raw[col] = (
            df_raw[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

    # Ano = primeiro ano do intervalo (ex: 2000/2001 → 2000)
    year_map = {c: int(c.split("/")[0]) for c in year_cols}
    df_raw = df_raw.rename(columns=year_map)
    years = sorted(year_map.values())

    result = {}
    for commodity in df_raw["Commodity"].unique():
        result[commodity] = {}
        sub = df_raw[df_raw["Commodity"] == commodity]
        for country in sub["Country"].unique():
            sub2 = sub[sub["Country"] == country].copy()
            sub2 = sub2.set_index("Attribute")[years]
            result[commodity][country] = sub2

    return result, years


def get_world_demand(data: dict, commodity: str, years: list) -> pd.Series:
    """Retorna série de demanda mundial (Domestic Consumption)."""
    try:
        attr = "Domestic Consumption" if commodity != "Sugar, Centrifugal" else "Total Disappearance"
        s = data[commodity]["World"].loc[attr, years]
        return s.apply(pd.to_numeric, errors="coerce")
    except Exception:
        return pd.Series(dtype=float)


def get_brazil_production(data: dict, commodity: str, years: list) -> pd.Series:
    """Retorna série de produção do Brasil."""
    try:
        s = data[commodity]["Brazil"].loc["Production", years]
        return s.apply(pd.to_numeric, errors="coerce")
    except Exception:
        return pd.Series(dtype=float)


def get_brazil_share(data: dict, commodity: str, years: list) -> pd.Series:
    """Participação do Brasil na produção mundial (%)."""
    br  = get_brazil_production(data, commodity, years)
    wld = data[commodity]["World"].loc["Production", years].apply(pd.to_numeric, errors="coerce")
    share = (br / wld * 100).replace([np.inf, -np.inf], np.nan)
    return share


# ──────────────────────────────────────────────────────────────────────────────
# 3. MONTE CARLO: DEMANDA
# ──────────────────────────────────────────────────────────────────────────────

def fit_growth_model(series: pd.Series):
    """
    Ajusta um modelo de crescimento à série (log-retornos).
    Retorna (mu, sigma) da distribuição normal dos retornos anuais.
    """
    s = series.dropna()
    if len(s) < 5:
        return 0.02, 0.05
    log_ret = np.log(s / s.shift(1)).dropna()
    mu    = log_ret.mean()
    sigma = log_ret.std()
    return float(mu), float(sigma)


def monte_carlo_demand(series: pd.Series, horizon: int = HORIZON, n_sim: int = N_SIMUL):
    """
    Simula caminhos futuros de demanda via GBM (Geometric Brownian Motion).
    Retorna array (n_sim, horizon).
    """
    mu, sigma = fit_growth_model(series)
    last_val  = series.dropna().iloc[-1]

    # GBM discreto: S(t+1) = S(t) * exp((mu - 0.5*sigma²) * dt + sigma * sqrt(dt) * Z)
    dt   = 1.0
    eps  = np.random.normal(0, 1, (n_sim, horizon))
    drift = (mu - 0.5 * sigma**2) * dt
    shocks = sigma * np.sqrt(dt) * eps

    log_returns = drift + shocks
    cum_returns = np.cumsum(log_returns, axis=1)
    paths = last_val * np.exp(cum_returns)
    return paths, mu, sigma


# ──────────────────────────────────────────────────────────────────────────────
# 4. MONTE CARLO: PREÇO
# ──────────────────────────────────────────────────────────────────────────────

def build_price_series(crop_label: str) -> pd.Series:
    """Constrói série histórica de preço para a cultura."""
    idx = PRICE_COLS.index(crop_label)
    data = {yr: vals[idx] for yr, vals in PRICES_HIST.items()}
    return pd.Series(data, name=crop_label)


def monte_carlo_price(crop_label: str, horizon: int = HORIZON, n_sim: int = N_SIMUL):
    """
    Simula preços futuros via GBM com reversão à média (Ornstein–Uhlenbeck)
    para commodities com comportamento mean-reverting.
    """
    s = build_price_series(crop_label)
    log_ret = np.log(s / s.shift(1)).dropna()

    mu_p    = float(log_ret.mean())
    sigma_p = float(log_ret.std())
    last_p  = float(s.iloc[-1])

    # Mean-reversion speed (kappa) estimado empiricamente
    kappa = 0.20  # velocidade de reversão moderada
    long_run_mean = float(np.exp(np.log(s).mean()))  # média log-normal de longo prazo

    dt = 1.0
    paths = np.zeros((n_sim, horizon))
    P = last_p
    price_arr = np.full(n_sim, P)

    for t in range(horizon):
        Z = np.random.normal(0, 1, n_sim)
        # OU discretizado
        price_arr = (
            price_arr
            + kappa * (long_run_mean - price_arr) * dt
            + sigma_p * price_arr * np.sqrt(dt) * Z
        )
        price_arr = np.clip(price_arr, 0.01, None)  # preço não negativo
        paths[:, t] = price_arr

    return paths, mu_p, sigma_p, long_run_mean


# ──────────────────────────────────────────────────────────────────────────────
# 5. ANÁLISE DO POTENCIAL DE MERCADO (1C do desafio)
# ──────────────────────────────────────────────────────────────────────────────

def calc_market_potential(demand_paths, price_paths, demand_unit_factor=1.0):
    """
    Potencial de mercado = Demanda (t) × Preço (US$/unidade).
    Retorna estatísticas por ano de projeção.
    """
    # demand_paths: (n_sim, horizon) em unidades originais
    # price_paths : (n_sim, horizon) em US$/unidade referência
    market = demand_paths * price_paths * demand_unit_factor  # em US$ (mesma unidade)

    result = []
    for t in range(demand_paths.shape[1]):
        arr = market[:, t]
        result.append({
            "year" : PROJ_YEAR + t + 1,
            "mean" : np.mean(arr),
            "median": np.median(arr),
            "p10"  : np.percentile(arr, 10),
            "p90"  : np.percentile(arr, 90),
            "std"  : np.std(arr),
        })
    return pd.DataFrame(result)


# ──────────────────────────────────────────────────────────────────────────────
# 6. SCORING PARA PRIORIZAÇÃO ESTRATÉGICA (1D do desafio)
# ──────────────────────────────────────────────────────────────────────────────

SCORING_WEIGHTS = {
    "tamanho_mercado"         : 0.25,  # potencial absoluto de mercado (Demanda × Preço)
    "crescimento_demanda"     : 0.20,  # CAGR histórico demanda mundial
    "posicao_brasil"          : 0.20,  # share do Brasil na produção mundial
    "tendencia_preco"         : 0.15,  # tendência de preço (positivo = atrativo)
    "volatilidade_preco"      : 0.10,  # menor volatilidade = menos risco
    "produtividade_brasil"    : 0.10,  # crescimento de yield do Brasil
}

def normalize_scores(raw: dict) -> dict:
    """Min-max normalization para escala 1-10."""
    vals = np.array(list(raw.values()), dtype=float)
    v_min, v_max = vals.min(), vals.max()
    if v_max == v_min:
        return {k: 5.0 for k in raw}
    return {k: 1 + 9 * (v - v_min) / (v_max - v_min) for k, v in raw.items()}


def build_scoring_table(data: dict, years: list, mc_results: dict) -> pd.DataFrame:
    """
    Constrói tabela de scoring multicritério para as 5 culturas.
    """
    rows = []
    raw_scores = {crit: {} for crit in SCORING_WEIGHTS}

    # 1. Tamanho de mercado: média do potencial projetado (ano 5)
    for crop_lbl in CROP_MAP.values():
        if crop_lbl in mc_results:
            mkt_mean = mc_results[crop_lbl]["market_potential"]["mean"].iloc[-1]
            raw_scores["tamanho_mercado"][crop_lbl] = mkt_mean

    # 2. Crescimento demanda: mu histórico (CAGR implícito)
    for crop_raw, crop_lbl in CROP_MAP.items():
        s = get_world_demand(data, crop_raw, years)
        mu, _ = fit_growth_model(s)
        raw_scores["crescimento_demanda"][crop_lbl] = mu

    # 3. Posição do Brasil: share médio últimos 5 anos
    for crop_raw, crop_lbl in CROP_MAP.items():
        share = get_brazil_share(data, crop_raw, years)
        raw_scores["posicao_brasil"][crop_lbl] = float(share.iloc[-5:].mean())

    # 4. Tendência de preço: slope linear normalizado
    for crop_lbl in CROP_MAP.values():
        ps = build_price_series(crop_lbl)
        x  = np.arange(len(ps))
        slope, *_ = np.polyfit(x, ps.values, 1)
        raw_scores["tendencia_preco"][crop_lbl] = slope / float(ps.mean())  # slope relativo

    # 5. Volatilidade preço: menor sigma = melhor (inverted)
    for crop_lbl in CROP_MAP.values():
        ps = build_price_series(crop_lbl)
        lr = np.log(ps / ps.shift(1)).dropna()
        raw_scores["volatilidade_preco"][crop_lbl] = -float(lr.std())  # negativo: menor vol = maior score

    # 6. Produtividade Brasil: slope yield (se disponível)
    for crop_raw, crop_lbl in CROP_MAP.items():
        try:
            yld = data[crop_raw]["Brazil"].loc["Yield", years].apply(pd.to_numeric, errors="coerce").dropna()
            if len(yld) >= 5:
                x = np.arange(len(yld))
                slope, *_ = np.polyfit(x, yld.values, 1)
                raw_scores["produtividade_brasil"][crop_lbl] = float(slope / yld.mean())
            else:
                raw_scores["produtividade_brasil"][crop_lbl] = 0.0
        except Exception:
            raw_scores["produtividade_brasil"][crop_lbl] = 0.0

    # Normaliza cada critério
    norm = {crit: normalize_scores(raw_scores[crit]) for crit in SCORING_WEIGHTS}

    # Score final ponderado
    crops = list(CROP_MAP.values())
    final_scores = {}
    for crop in crops:
        score = sum(SCORING_WEIGHTS[crit] * norm[crit].get(crop, 5.0)
                    for crit in SCORING_WEIGHTS)
        final_scores[crop] = score

    # Monta DataFrame detalhado
    records = []
    for crop in crops:
        row = {"Cultura": crop}
        for crit in SCORING_WEIGHTS:
            row[crit] = round(norm[crit].get(crop, 5.0), 2)
        row["Score Final"] = round(final_scores[crop], 2)
        records.append(row)

    df = pd.DataFrame(records).sort_values("Score Final", ascending=False)
    df["Ranking"] = range(1, len(df) + 1)
    return df


# ──────────────────────────────────────────────────────────────────────────────
# 7. VISUALIZAÇÕES
# ──────────────────────────────────────────────────────────────────────────────

LEK_GREEN  = "#00A651"
LEK_DKGRN  = "#004D2C"
LEK_LGRAY  = "#D0D0D0"
LEK_GRAY   = "#555555"
BG_COLOR   = "#FAFAFA"
PALETTE    = ["#00A651", "#007D3F", "#33BF77", "#99DFB9", "#CCF0DC"]

def plot_mc_demand(crop_lbl: str, demand_paths: np.ndarray, hist_series: pd.Series, ax):
    """Fan chart de demanda futura."""
    proj_years = list(range(PROJ_YEAR + 1, PROJ_YEAR + HORIZON + 1))
    p10 = np.percentile(demand_paths, 10, axis=0)
    p25 = np.percentile(demand_paths, 25, axis=0)
    p75 = np.percentile(demand_paths, 75, axis=0)
    p90 = np.percentile(demand_paths, 90, axis=0)
    med = np.median(demand_paths, axis=0)

    # Histórico
    hist_x = list(hist_series.index)
    ax.plot(hist_x, hist_series.values, color=LEK_DKGRN, lw=2, label="Histórico")

    # Fan
    ax.fill_between(proj_years, p10, p90, alpha=0.20, color=LEK_GREEN, label="P10–P90")
    ax.fill_between(proj_years, p25, p75, alpha=0.35, color=LEK_GREEN, label="P25–P75")
    ax.plot(proj_years, med, color=LEK_GREEN, lw=2.5, ls="--", label="Mediana")

    ax.set_title(f"{crop_lbl} – Demanda Mundial", fontsize=11, fontweight="bold", color=LEK_DKGRN)
    ax.set_xlabel("Ano", fontsize=9)
    ax.set_facecolor(BG_COLOR)
    ax.spines[["top", "right"]].set_visible(False)


def plot_mc_price(crop_lbl: str, price_paths: np.ndarray, price_hist: pd.Series, ax):
    """Fan chart de preço futuro."""
    proj_years = list(range(PROJ_YEAR + 1, PROJ_YEAR + HORIZON + 1))
    p10 = np.percentile(price_paths, 10, axis=0)
    p25 = np.percentile(price_paths, 25, axis=0)
    p75 = np.percentile(price_paths, 75, axis=0)
    p90 = np.percentile(price_paths, 90, axis=0)
    med = np.median(price_paths, axis=0)

    ax.plot(list(price_hist.index), price_hist.values, color=LEK_DKGRN, lw=2, label="Histórico")
    ax.fill_between(proj_years, p10, p90, alpha=0.20, color="#2196F3", label="P10–P90")
    ax.fill_between(proj_years, p25, p75, alpha=0.35, color="#2196F3", label="P25–P75")
    ax.plot(proj_years, med, color="#1565C0", lw=2.5, ls="--", label="Mediana")

    u = UNITS[crop_lbl]["price"]
    ax.set_title(f"{crop_lbl} – Preço ({u})", fontsize=11, fontweight="bold", color=LEK_DKGRN)
    ax.set_xlabel("Ano", fontsize=9)
    ax.set_facecolor(BG_COLOR)
    ax.spines[["top", "right"]].set_visible(False)


def plot_scoring(df_score: pd.DataFrame, ax):
    """Horizontal bar chart do scoring final."""
    colors = [LEK_GREEN if i < 2 else LEK_LGRAY for i in range(len(df_score))]
    bars = ax.barh(df_score["Cultura"], df_score["Score Final"], color=colors, edgecolor="white")
    ax.set_xlim(0, 10)
    ax.set_xlabel("Score Ponderado (0–10)", fontsize=9)
    ax.set_title("Priorização Estratégica – Scoring Multicritério", fontsize=11,
                 fontweight="bold", color=LEK_DKGRN)
    ax.invert_yaxis()

    for bar, val in zip(bars, df_score["Score Final"]):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
                f"{val:.2f}", va="center", fontsize=9, color=LEK_GRAY)

    # Destaque top 2
    ax.text(0.98, 0.05, "★ Top 2 priorizadas", transform=ax.transAxes,
            ha="right", va="bottom", color=LEK_GREEN, fontsize=9, fontstyle="italic")
    ax.set_facecolor(BG_COLOR)
    ax.spines[["top", "right"]].set_visible(False)


def plot_brazil_share(data: dict, years: list, ax):
    """Evolução do share do Brasil na produção mundial."""
    for i, (crop_raw, crop_lbl) in enumerate(CROP_MAP.items()):
        share = get_brazil_share(data, crop_raw, years)
        share = share.dropna()
        ax.plot(share.index, share.values, lw=2, label=crop_lbl,
                color=PALETTE[i % len(PALETTE)], marker="o", markersize=3)

    ax.set_title("Share do Brasil na Produção Mundial (%)", fontsize=11,
                 fontweight="bold", color=LEK_DKGRN)
    ax.set_xlabel("Ano")
    ax.set_ylabel("%")
    ax.legend(fontsize=8, loc="upper left")
    ax.set_facecolor(BG_COLOR)
    ax.spines[["top", "right"]].set_visible(False)


def plot_market_potential(mc_results: dict, ax):
    """Barras de potencial de mercado (mediana ano 5)."""
    crops, vals_med, vals_p25, vals_p75 = [], [], [], []
    for crop_lbl, res in mc_results.items():
        mp = res["market_potential"]
        crops.append(crop_lbl)
        vals_med.append(mp["median"].iloc[-1])
        vals_p25.append(mp["p10"].iloc[-1])
        vals_p75.append(mp["p90"].iloc[-1])

    x = np.arange(len(crops))
    bars = ax.bar(x, vals_med, color=LEK_GREEN, alpha=0.85, edgecolor="white")
    ax.errorbar(x, vals_med,
                yerr=[np.array(vals_med) - np.array(vals_p25),
                      np.array(vals_p75) - np.array(vals_med)],
                fmt="none", color=LEK_DKGRN, capsize=5, lw=1.5)
    ax.set_xticks(x)
    ax.set_xticklabels(crops, rotation=20, ha="right", fontsize=8)
    ax.set_title(f"Potencial de Mercado – Mediana ({PROJ_YEAR+HORIZON}) [unid. orig. × preço]",
                 fontsize=10, fontweight="bold", color=LEK_DKGRN)
    ax.set_facecolor(BG_COLOR)
    ax.spines[["top", "right"]].set_visible(False)


# ──────────────────────────────────────────────────────────────────────────────
# 8. PIPELINE PRINCIPAL
# ──────────────────────────────────────────────────────────────────────────────

def run_pipeline():
    print("=" * 65)
    print("  DESAFIO L.E.K. 2026 – Monte Carlo: Demanda & Preço")
    print("=" * 65)

    # ── 8.1 Carrega e trata base ─────────────────────────────────────────────
    print("\n[1/5] Carregando e tratando basezona.xlsx…")
    data, years = load_and_clean_base(INPUT_FILE)

    # Filtra histórico até 2023 (exclui projeções 2024/25 do USDA)
    years_hist = [y for y in years if y <= 2023]

    print(f"      Culturas encontradas: {list(CROP_MAP.keys())}")
    print(f"      Período histórico: {years_hist[0]} – {years_hist[-1]}")

    # ── 8.2 Monte Carlo por cultura ──────────────────────────────────────────
    print("\n[2/5] Rodando Monte Carlo (demanda + preço)…")
    mc_results = {}

    for crop_raw, crop_lbl in CROP_MAP.items():
        print(f"      → {crop_lbl}")
        demand_hist = get_world_demand(data, crop_raw, years_hist).dropna()
        price_hist  = build_price_series(crop_lbl)

        # Demanda
        demand_paths, mu_d, sigma_d = monte_carlo_demand(demand_hist)

        # Preço
        price_paths, mu_p, sigma_p, lr_mean = monte_carlo_price(crop_lbl)

        # Potencial de mercado
        # Fator de conversão aproximado para tornar compatível:
        # (demanda em unidade original) × (preço em US$) → índice comparativo
        mkt = calc_market_potential(demand_paths, price_paths)

        mc_results[crop_lbl] = {
            "demand_paths"    : demand_paths,
            "price_paths"     : price_paths,
            "demand_hist"     : demand_hist,
            "price_hist"      : price_hist,
            "mu_demand"       : mu_d,
            "sigma_demand"    : sigma_d,
            "mu_price"        : mu_p,
            "sigma_price"     : sigma_p,
            "long_run_price"  : lr_mean,
            "market_potential": mkt,
        }

    # ── 8.3 Scoring multicritério ─────────────────────────────────────────────
    print("\n[3/5] Calculando scoring multicritério…")
    df_score = build_scoring_table(data, years_hist, mc_results)

    print("\n  ┌─── RESULTADO DO SCORING ───────────────────────────────────┐")
    for _, row in df_score.iterrows():
        star = " ★" if row["Ranking"] <= 2 else "  "
        print(f"  │ #{int(row['Ranking'])} {row['Cultura']:<22} Score: {row['Score Final']:.2f}/10{star}")
    print("  └────────────────────────────────────────────────────────────┘")
    top2 = df_score[df_score["Ranking"] <= 2]["Cultura"].tolist()
    print(f"\n  ✅ Culturas priorizadas: {top2[0]} e {top2[1]}")

    # ── 8.4 Geração dos gráficos ──────────────────────────────────────────────
    print("\n[4/5] Gerando visualizações…")

    crops = list(CROP_MAP.values())
    n = len(crops)

    # ── Fig 1: Fan charts de demanda ──
    fig1, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig1.patch.set_facecolor(BG_COLOR)
    fig1.suptitle("Monte Carlo – Demanda Mundial por Cultura (10.000 simulações)",
                  fontsize=14, fontweight="bold", color=LEK_DKGRN, y=1.01)

    for i, crop_lbl in enumerate(crops):
        ax = axes[i // 3][i % 3]
        res = mc_results[crop_lbl]
        plot_mc_demand(crop_lbl, res["demand_paths"], res["demand_hist"], ax)
        if i == 0:
            ax.legend(fontsize=8, loc="upper left")

    # Esconde eixo extra
    axes[1][2].set_visible(False)
    plt.tight_layout()
    fig1.savefig(OUTPUT_DIR / "fig1_demand_mc.png", dpi=150, bbox_inches="tight",
                 facecolor=BG_COLOR)
    plt.close(fig1)

    # ── Fig 2: Fan charts de preço ──
    fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10))
    fig2.patch.set_facecolor(BG_COLOR)
    fig2.suptitle("Monte Carlo – Preço por Cultura (Ornstein-Uhlenbeck Mean Reversion)",
                  fontsize=14, fontweight="bold", color=LEK_DKGRN, y=1.01)

    for i, crop_lbl in enumerate(crops):
        ax = axes2[i // 3][i % 3]
        res = mc_results[crop_lbl]
        plot_mc_price(crop_lbl, res["price_paths"], res["price_hist"], ax)
        if i == 0:
            ax.legend(fontsize=8, loc="upper left")

    axes2[1][2].set_visible(False)
    plt.tight_layout()
    fig2.savefig(OUTPUT_DIR / "fig2_price_mc.png", dpi=150, bbox_inches="tight",
                 facecolor=BG_COLOR)
    plt.close(fig2)

    # ── Fig 3: Dashboard resumo ──
    fig3, axes3 = plt.subplots(2, 2, figsize=(16, 12))
    fig3.patch.set_facecolor(BG_COLOR)
    fig3.suptitle("L.E.K. Consulting 2026 – Dashboard de Análise Estratégica",
                  fontsize=15, fontweight="bold", color=LEK_DKGRN)

    plot_scoring(df_score, axes3[0][0])
    plot_market_potential(mc_results, axes3[0][1])
    plot_brazil_share(data, years_hist, axes3[1][0])

    # Tabela de parâmetros MC
    ax_tbl = axes3[1][1]
    ax_tbl.axis("off")
    tbl_data = []
    for crop_lbl in crops:
        res = mc_results[crop_lbl]
        tbl_data.append([
            crop_lbl,
            f"{res['mu_demand']*100:.1f}%",
            f"{res['sigma_demand']*100:.1f}%",
            f"{res['mu_price']*100:.1f}%",
            f"{res['sigma_price']*100:.1f}%",
        ])

    tbl = ax_tbl.table(
        cellText=tbl_data,
        colLabels=["Cultura", "μ Demanda", "σ Demanda", "μ Preço", "σ Preço"],
        loc="center", cellLoc="center"
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1, 1.8)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor(LEK_DKGRN)
            cell.set_text_props(color="white", fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#E8F5E9")
        cell.set_edgecolor("#CCCCCC")
    ax_tbl.set_title("Parâmetros do Modelo Monte Carlo", fontsize=10,
                     fontweight="bold", color=LEK_DKGRN, pad=20)

    plt.tight_layout()
    fig3.savefig(OUTPUT_DIR / "fig3_dashboard.png", dpi=150, bbox_inches="tight",
                 facecolor=BG_COLOR)
    plt.close(fig3)

    # ── Fig 4: Histograma de potencial de mercado para top 2 ──
    fig4, axes4 = plt.subplots(1, 2, figsize=(14, 6))
    fig4.patch.set_facecolor(BG_COLOR)
    fig4.suptitle(f"Distribuição do Potencial de Mercado – Top 2 Culturas (ano {PROJ_YEAR+HORIZON})",
                  fontsize=13, fontweight="bold", color=LEK_DKGRN)

    for i, crop_lbl in enumerate(top2):
        ax = axes4[i]
        res = mc_results[crop_lbl]
        mkt_final = res["demand_paths"][:, -1] * res["price_paths"][:, -1]
        ax.hist(mkt_final, bins=80, color=PALETTE[i], edgecolor="white", alpha=0.85)
        mu_m  = np.mean(mkt_final)
        p10_m = np.percentile(mkt_final, 10)
        p90_m = np.percentile(mkt_final, 90)
        ax.axvline(mu_m,  color=LEK_DKGRN, lw=2.5, ls="-",  label=f"Média: {mu_m:,.0f}")
        ax.axvline(p10_m, color=LEK_GRAY,  lw=1.5, ls="--", label=f"P10: {p10_m:,.0f}")
        ax.axvline(p90_m, color=LEK_GRAY,  lw=1.5, ls="--", label=f"P90: {p90_m:,.0f}")
        ax.set_title(f"{crop_lbl}", fontsize=11, fontweight="bold", color=LEK_DKGRN)
        ax.set_xlabel("Potencial (unid. orig. × preço)", fontsize=9)
        ax.legend(fontsize=8)
        ax.set_facecolor(BG_COLOR)
        ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    fig4.savefig(OUTPUT_DIR / "fig4_top2_distribution.png", dpi=150, bbox_inches="tight",
                 facecolor=BG_COLOR)
    plt.close(fig4)

    # ── 8.5 Exporta resultados numéricos ────────────────────────────────────
    print("\n[5/5] Exportando resultados…")

    # CSV de scoring
    df_score.to_csv(OUTPUT_DIR / "scoring_priorizacao.csv", index=False, encoding="utf-8-sig")

    # CSV de projeções por cultura
    all_proj = []
    for crop_lbl, res in mc_results.items():
        for t in range(HORIZON):
            yr = PROJ_YEAR + t + 1
            dp = res["demand_paths"][:, t]
            pp = res["price_paths"][:, t]
            all_proj.append({
                "Cultura": crop_lbl,
                "Ano"    : yr,
                "Demanda_Mediana" : np.median(dp),
                "Demanda_P10"     : np.percentile(dp, 10),
                "Demanda_P90"     : np.percentile(dp, 90),
                "Demanda_Std"     : np.std(dp),
                "Preco_Mediana"   : np.median(pp),
                "Preco_P10"       : np.percentile(pp, 10),
                "Preco_P90"       : np.percentile(pp, 90),
                "Preco_Std"       : np.std(pp),
                "Potencial_Mediana": res["market_potential"]["median"].iloc[t],
                "Potencial_P10"   : res["market_potential"]["p10"].iloc[t],
                "Potencial_P90"   : res["market_potential"]["p90"].iloc[t],
            })

    df_proj = pd.DataFrame(all_proj)
    df_proj.to_csv(OUTPUT_DIR / "projecoes_mc.csv", index=False, encoding="utf-8-sig",
                   float_format="%.4f")

    # Resumo JSON (para uso no slide / apresentação)
    summary = {
        "top2": top2,
        "parametros": {
            crop_lbl: {
                "mu_demanda_pct"  : round(res["mu_demand"] * 100, 2),
                "sigma_demanda_pct": round(res["sigma_demand"] * 100, 2),
                "mu_preco_pct"    : round(res["mu_price"] * 100, 2),
                "sigma_preco_pct" : round(res["sigma_price"] * 100, 2),
                "preco_long_run"  : round(res["long_run_price"], 4),
            }
            for crop_lbl, res in mc_results.items()
        },
        "scoring": df_score[["Cultura", "Score Final", "Ranking"]].to_dict("records"),
    }
    with open(OUTPUT_DIR / "resumo_mc.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    # ── Sumário final ────────────────────────────────────────────────────────
    print("\n" + "=" * 65)
    print("  SUMÁRIO DOS ARQUIVOS GERADOS")
    print("=" * 65)
    outputs = [
        ("fig1_demand_mc.png",        "Fan charts – Demanda (MC)"),
        ("fig2_price_mc.png",         "Fan charts – Preço (MC Ornstein-Uhlenbeck)"),
        ("fig3_dashboard.png",        "Dashboard estratégico completo"),
        ("fig4_top2_distribution.png","Distribuições do potencial de mercado (Top 2)"),
        ("scoring_priorizacao.csv",   "Tabela de scoring multicritério"),
        ("projecoes_mc.csv",          "Projeções quantiladas (P10/Mediana/P90)"),
        ("resumo_mc.json",            "Resumo estruturado para slides"),
    ]
    for fname, desc in outputs:
        path = OUTPUT_DIR / fname
        exists = "✓" if path.exists() else "✗"
        print(f"  {exists} {fname:<40} {desc}")

    print("\n" + "=" * 65)
    print(f"  ✅ Top 2 recomendadas: {top2[0]} e {top2[1]}")
    print("  Simulações: {:,} | Horizonte: {} anos | Modelo: GBM + OU".format(N_SIMUL, HORIZON))
    print("=" * 65)

    return mc_results, df_score, df_proj, summary


# ──────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    mc_results, df_score, df_proj, summary = run_pipeline()


  DESAFIO L.E.K. 2026 – Monte Carlo: Demanda & Preço

[1/5] Carregando e tratando basezona.xlsx…
      Culturas encontradas: ['Corn', 'Coffee, Green', 'Sugar, Centrifugal', 'Orange Juice', 'Wheat']
      Período histórico: 2000 – 2023

[2/5] Rodando Monte Carlo (demanda + preço)…
      → Milho
      → Café (Arábica)
      → Cana-de-açúcar
      → Citros (OJ)
      → Trigo

[3/5] Calculando scoring multicritério…

  ┌─── RESULTADO DO SCORING ───────────────────────────────────┐
  │ #1 Citros (OJ)            Score: 6.70/10 ★
  │ #2 Café (Arábica)         Score: 5.95/10 ★
  │ #3 Milho                  Score: 3.17/10  
  │ #4 Cana-de-açúcar         Score: 3.06/10  
  │ #5 Trigo                  Score: 2.54/10  
  └────────────────────────────────────────────────────────────┘

  ✅ Culturas priorizadas: Citros (OJ) e Café (Arábica)

[4/5] Gerando visualizações…

[5/5] Exportando resultados…

  SUMÁRIO DOS ARQUIVOS GERADOS
  ✓ fig1_demand_mc.png                       Fan charts – Demanda (MC)